package a installer dans l"env : 
* django 
* psycopg2
* pandas
* numpy
* jupyterlab
* openpyxl
* SQLAlchemy

# import package

In [1]:
import pandas as pd 
import numpy as np 
import psycopg2 as pg
import datetime as dt
import warnings
import re
import string 
import ast
from datetime import datetime
from sqlalchemy import create_engine
import os 

warnings.filterwarnings("ignore")

# creation fonction utile

In [2]:
HOST = "localhost"
USERNAME = "postgres"
PASSWORD = "0000"
PORT = "5432"
DBNAME = "obcast_project"

In [3]:
def create_connection():
    conn = pg.connect(
        database=DBNAME,
        host=HOST,
        user=USERNAME,
        password=PASSWORD,
        port=PORT
    )
    return conn

conn = create_connection()


In [4]:
def execute_query(query, conn = conn):
    cursor = conn.cursor()
    cursor.execute(query)
    cursor.close()
    conn.commit()

# lecture excel et nettoyage

In [5]:
df = pd.read_excel("data/Bdd_Obcast_2025.xlsx")
df.head(3)

,STATUT DU PRODUCTEUR,TYPE DE PRODUCTEUR,NOM DU PRODUCTEUR,Studio de production,coproducteurs/partenariats,URL,flux RSS,NOM,COLLECTION,Résumé officiel,...,ACPM - FEVRIER 2022 - TÉLÉCHARGEMENTS MONDE SUR UN MOIS,SPOTIFY ETIQUETTES THEMATIQUES,APPLE PODCAST ETIQUETTES THEMATIQUES,PRIX /RECOMPENSES,DATE DE LA Première COLLECTE,Date de la deuxième collecte,pod_id_ArteRadio_INA_matchéauto,ABS INA,url pour collecte INA,NOTES COMPLEMENTAIRES
0,Groupe média public,TV,Arte Radio,non indiqué,non indiqué,https://www.arteradio.com/serie/hommage_stanle...,NaN,Hommage à Stanley Kubrick,À suivre,7 reportages et créations en hommage à Kubrick,...,non,non,non,nsp,2022-03-01 00:00:00,2024-01-17 00:00:00,hommage_a_stanley_kubrick,NaN,hommage_stanley_kubrick,NB : nombre d'épisode pour le média pas représ...
1,Groupe média public,TV,Arte Radio,non indiqué,non indiqué,https://www.arteradio.com/serie/journal_d_une_...,NaN,Journal d'une jeune prof,nsp,Jeune professeur de lettres dans un collège de...,...,AD,ne s'applique pas,nsp,NaN,ne s'applique pas,2024-01-17 00:00:00,journal_d_une_jeune_prof,NaN,journal_d_une_jeune_prof,NaN
2,Groupe média public,TV,Arte Radio,non indiqué,non indiqué,https://www.arteradio.com/serie/assa_jeune_fil...,NaN,"Assa, jeune fille de cité",nsp,Assa Diakité vit dans une cité de la banlieue ...,...,AD,ne s'applique pas,ne s'applique pas,NaN,ne s'applique pas,2024-01-17 00:00:00,assa_jeune_fille_de_cite,NaN,assa_jeune_fille_de_cite,NaN


# peuplement 

## mot cle

In [6]:
df["Mots-clefs"].value_counts()

Mots-clefs
non                              610
musique                           80
cinéma                            49
féminisme?                        39
arts, médias                      35
                                ... 
prison, ruralité                   1
Israel, Palestine, banlieue        1
éducation, numérique               1
mémoire, témoignage                1
aventure ; randonnée ; sport       1
Name: count, Length: 864, dtype: int64

In [7]:
motscle = df["Mots-clefs"].dropna().unique()
motscle

array(['cinéma', 'éducation', 'témoignage', 'création sonore, Paris',
       'littérature, poésie', 'création sonore, voyage, patrimoine',
       'justice, procès, terrorisme', 'éducation, témoignage', 'banlieue',
       'immigration', 'maternité, féminin, féminisme',
       'drogue, témoignage', 'élection, abstention', 'création sonore',
       'non', 'banlieu, éducation', 'histoire, télévision', 'islam',
       'média', 'afrique, cinéma, politique', 'sexualité, parentalité',
       'santé mentale, psychiatrie', 'musique', 'photographie',
       'international, Afrique; culture', 'musique, humour',
       'jeux vidéos', 'voiture, voyage', 'littérature', 'télévision',
       'divertissement', 'amitié, intimité', 'animalité, création sonore',
       'adolescence', 'police, xénophobie', 'sexualité',
       'témoignage, intimité', 'divertissement, technologie', 'jeunesse',
       'musique, rap', 'théâtre, cinéma', 'toxicomanie, témoignage',
       'cinéma, télévision', 'éducation, témoign

In [8]:
motscle_clean = [] 
for i in motscle : 
    # print(i)
    motscle_clean.append(re.sub(f"[{re.escape(string.punctuation)}]+", ",", i))

mot_cle_f = []
for i in motscle_clean :
    i = i.split(",")
    i = [x.strip().lower() for x in i if len(x.strip()) > 1]
    mot_cle_f += i

print("avant set : ", len(mot_cle_f))
mot_cle_f = list(set(mot_cle_f))
print("apres set : ", len(mot_cle_f))
mot_cle_f.remove("non")
print("apres remove : ", len(mot_cle_f))
mot_cle_f = sorted(mot_cle_f)

avant set :  1708
apres set :  461
apres remove :  460


In [9]:
query = """
    INSERT INTO main_motcle(label) VALUES 
"""
for i in mot_cle_f : 
    query += f"('{i}'),"

query = query[:-1] + ";"

try : 
    execute_query(query, conn)
except : 
    conn = create_connection()
    execute_query(query, conn)

## Rubrique

In [10]:
df["Rubrique"].value_counts()

Rubrique
productions culturelles              485
social/société                       388
sport                                178
économie/emploi                      157
tranches de vie                      147
faits divers/justice                 128
vie quotidienne/vie pratique         116
histoire                             111
politique                            109
santé                                 98
actualité générale                    94
écologie/nature                       89
sciences                              86
gastronomie, tourisme, patrimoine     75
vie politique                         69
journalisme                           69
religion                              27
Histoire                               7
culture                                3
Name: count, dtype: int64

In [11]:
rubrique = df["Rubrique"].dropna().unique()


rubrique_clean = [] 
for i in rubrique : 
    rubrique_clean.append(re.sub(f"[{re.escape(string.punctuation)}]+", ",", i))

rubrique_f = []
for i in rubrique_clean :
    i = i.split(",")
    i = [x.strip().lower() for x in i if len(x.strip()) > 1]
    rubrique_f += i

print("avant set : ", len(rubrique_f))
rubrique_f = list(set(rubrique_f))
print("apres set : ", len(rubrique_f))

query = """
    INSERT INTO main_rubrique(label) VALUES 
"""
for i in rubrique_f : 
    query += f"('{i}'),"

query = query[:-1] + ";"

try : 
    execute_query(query, conn)
except : 
    conn = create_connection()
    execute_query(query, conn)

avant set :  26
apres set :  25


## recompense

In [12]:
recompense = list({x.strip().lower().replace("'", "''") for x in df["PRIX /RECOMPENSES"].dropna().unique()})
recompense

['nsp',
 'non',
 'non précisé',
 'prix scam du podcast documentaire 2021 - paris podcast festival',
 'ad',
 'prix du meilleur podcast jeunesse au paris podcast festival 2021.',
 "mention spéciale festival longueurs d''onde, 2021"]

In [13]:
query = """
    INSERT INTO main_recompense(label) VALUES 
"""
for i in recompense : 
    query += f"('{i}'),"

query = query[:-1] + ";"

try : 
    execute_query(query, conn)
except : 
    conn = create_connection()
    execute_query(query, conn)

## Genre format

In [14]:
df["Genre/format journalistique"].value_counts()

Genre/format journalistique
Documentaire/reportage      1015
Interview                    551
Chronique                    262
Émission                     150
Enrobé                       127
Débat                         81
enrobé                        80
Autre / variable              42
Portrait                      39
Captation/Retransmission      30
débat                         27
émission                      11
chronique                     10
interview                     10
Récit                          1
Name: count, dtype: int64

In [15]:
genre = df["Genre/format journalistique"].dropna().unique()


genre_clean = [] 
for i in genre : 
    genre_clean.append(re.sub(f"[{re.escape(string.punctuation)}]+", ",", i))

genre_f = []
for i in genre_clean :
    i = i.split(",")
    i = [x.strip().lower() for x in i if len(x.strip()) > 1]
    genre_f += i

print("avant set : ", len(genre_f))
genre_f = list(set(genre_f))
print("apres set : ", len(genre_f))

query = """
    INSERT INTO main_genre(label) VALUES 
"""
for i in genre_f : 
    query += f"('{i}'),"

query = query[:-1] + ";"

try : 
    execute_query(query, conn)
except : 
    conn = create_connection()
    execute_query(query, conn)

avant set :  18
apres set :  13


## plateforme

In [16]:
# ne pas de unpivot le df au moment de la recuperartion des valeurs pour la table de jointure
plateforme =  [x.split("disponible sur")[0].strip() for x in df.columns if "disponible sur" in x]
plateforme

['SITE DU PRODUCTEUR',
 'APPLE PODCAST',
 'SPOTIFY',
 'DEEZER',
 'GOOGLE PODCAST',
 'CASTBOX',
 'PODCAST ADDICT',
 'YOUTUBE',
 'SOUNDCLOUD',
 'SITE PARTENAIRE',
 'STITCHER',
 'AMAZON MUSIC']

In [17]:
query = """
    INSERT INTO main_plateforme(label) VALUES 
"""
for i in plateforme : 
    query += f"('{i}'),"

query = query[:-1] + ";"

try : 
    execute_query(query, conn)
except : 
    conn = create_connection()
    execute_query(query, conn)

## etiquette

In [18]:
# cas spotify : 
df["SPOTIFY ETIQUETTES THEMATIQUES"].value_counts()

SPOTIFY ETIQUETTES THEMATIQUES
ne s'applique pas                                    657
non                                                  495
pas d'étiquette                                      107
Société, culture                                      96
société, documentaire, divertissement                 68
                                                    ... 
design et architecture                                 1
société, culture, philosophie                          1
Affaires, educational podcasts, carrière               1
Relations                                              1
affaire criminelle, récits intimes et témoignages      1
Name: count, Length: 382, dtype: int64

In [19]:
def get_singulier(word) : 
    if word in ["les", "des", "deux", "fois", "mois", "pas", "tennis"] : 
        return word
    elif word.endswith("s") or word.endswith("x") : 
        return word[:-1]
    return word

    
etiquette = df["SPOTIFY ETIQUETTES THEMATIQUES"].dropna().unique()
etiquette = {i : i for i in etiquette}

for i, j in etiquette.items() : 
    etiquette[i] = re.sub(f"[{re.escape(string.punctuation)}]+", ",", j)
    
etiquette["non"] = "non"
etiquette["ne s'applique pas"] = "ne s'applique pas"

etiquette_f = {}

for idx, i in etiquette.items():
    i = i.split(",")
    i = [x.strip().lower() for x in i if len(x.strip()) > 1]
    for k, v in enumerate(i) : 
        etiquette_f[(idx, k)] = v

for idx, i in etiquette_f.items(): 
    i = i.strip().lower().split(" ")
    i = " ".join([get_singulier(x) for x in i])
    etiquette_f[idx] = i

final = sorted(list(set(etiquette_f.values())))
query = """
    INSERT INTO main_etiquette(label, type_etiquette) VALUES 
"""
for i in final : 
    i = i.replace("'", "''")
    query += f"('{i}', 'SPOTIFY'),"

query = query[:-1] + ";"

try : 
    execute_query(query, conn)
except : 
    conn = create_connection()
    execute_query(query, conn)

In [20]:
etiquette = df["APPLE PODCAST ETIQUETTES THEMATIQUES"].dropna().unique()
etiquette = {i : i for i in etiquette}

for i, j in etiquette.items() : 
    etiquette[i] = re.sub(f"[{re.escape(string.punctuation)}]+", ",", j)
    
etiquette["non"] = "non"
etiquette["ne s'applique pas"] = "ne s'applique pas"

etiquette_f = {}

for idx, i in etiquette.items():
    i = i.split(",")
    i = [x.strip().lower() for x in i if len(x.strip()) > 1]
    for k, v in enumerate(i) : 
        etiquette_f[(idx, k)] = v

for idx, i in etiquette_f.items(): 
    i = i.strip().lower().split(" ")
    i = " ".join([get_singulier(x) for x in i])
    etiquette_f[idx] = i

final = sorted(list(set(etiquette_f.values())))
query = """
    INSERT INTO main_etiquette(label, type_etiquette) VALUES 
"""
for i in final : 
    i = i.replace("'", "''")
    query += f"('{i}', 'APPLE'),"

query = query[:-1] + ";"

try : 
    execute_query(query, conn)
except : 
    conn = create_connection()
    execute_query(query, conn)

## Createur

In [21]:
df_createur = pd.read_excel("createur_extract.xlsx")
df_createur["index"] = df_createur.index
df_createur.head()

,Créateur/auteur : Nom et informations,Catégorie Créateur/auteur,Genre de l'host,NOM DE L'HOST,Extracted Info,index
0,"Joseph Beauregard documentariste, journalste, ...",auteur/réalisateur,H+F,AD,"[('Joseph Beauregard', 'male'), ('journalste',...",0
1,Delphine Saltel,auteur/réalisateur,F,Delphine Saltel,"[('Delphine Saltel', 'female')]",1
2,Delphine Saltel,auteur/réalisateur,F,Delphine Saltel,"[('Delphine Saltel', 'female')]",2
3,Anthony Carcone,auteur/réalisateur,H,Anthony Carcone,"[('Anthony Carcone', 'male')]",3
4,Thomas Guillaud-Bataille,auteur/réalisateur,H,Thomas Guillaud-Bataille,"[('Thomas Guillaud-Bataille', 'male')]",4


In [22]:
df_createur["Extracted Info"] = df_createur["Extracted Info"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [23]:
createur = {"nom" : [], "genre" : [], "categorie" : [], "origin" : [], "index" : []}
for i, row in df_createur.iterrows():
    if len(row["Extracted Info"]) < 1:
        createur["nom"].append(row["Créateur/auteur : Nom et informations"])
        createur["genre"].append(row["Genre de l'host"])
        createur["categorie"].append(row["Catégorie Créateur/auteur"])
        createur["origin"].append(row["Créateur/auteur : Nom et informations"])
        createur["index"].append(i)
    else : 
        for personne in row["Extracted Info"]: 
            createur["nom"].append(personne[0])
            createur["genre"].append({"male" : "H", "female" : "F"}.get(personne[1], row["Genre de l'host"]))
            createur["categorie"].append(row["Catégorie Créateur/auteur"])
            createur["origin"].append(row["Créateur/auteur : Nom et informations"])
            createur["index"].append(i)
        

df_list_createur = pd.DataFrame(createur)
df_list_createur.head()

,nom,genre,categorie,origin,index
0,Joseph Beauregard,H,auteur/réalisateur,"Joseph Beauregard documentariste, journalste, ...",0
1,journalste,H+F,auteur/réalisateur,"Joseph Beauregard documentariste, journalste, ...",0
2,David Chowmentoski,H,auteur/réalisateur,"Joseph Beauregard documentariste, journalste, ...",0
3,Jérémi Nureni,H,auteur/réalisateur,"Joseph Beauregard documentariste, journalste, ...",0
4,Tatjana Bogucz,H+F,auteur/réalisateur,"Joseph Beauregard documentariste, journalste, ...",0


In [24]:
df_list_createur_unique = df_list_createur.drop_duplicates(subset=["nom", "genre"])

In [25]:
df_list_createur_unique.shape

(1725, 5)

In [26]:
query = """
    INSERT INTO main_createur(nom, genre, categorie_createur, origin) VALUES 
"""
for i, row in df_list_createur_unique.iterrows(): 
    query += f"""('{str(row["nom"]).replace("'", "''")}', '{str(row["genre"]).replace("'", "''")}', '{str(row["categorie"]).replace("'", "''")}', '{str(row["origin"]).replace("'", "''")}'),"""

query = query[:-1] + ";"

try : 
    execute_query(query, conn)
except : 
    conn = create_connection()
    execute_query(query, conn)


## producteur

In [27]:
df_collection = df[["NOM DU PRODUCTEUR", "STATUT DU PRODUCTEUR", "TYPE DE PRODUCTEUR", "Modèle économique", "NATIF/ENRICHI"]]
print(df_collection.shape)
df_collection.head()

(2436, 5)


,NOM DU PRODUCTEUR,STATUT DU PRODUCTEUR,TYPE DE PRODUCTEUR,Modèle économique,NATIF/ENRICHI
0,Arte Radio,Groupe média public,TV,gratuit,NATIF
1,Arte Radio,Groupe média public,TV,AD,AD
2,Arte Radio,Groupe média public,TV,AD,AD
3,Arte Radio,Groupe média public,TV,AD,AD
4,Arte Radio,Groupe média public,TV,AD,AD


In [28]:
df_collection = df_collection.drop_duplicates(subset=["NOM DU PRODUCTEUR", "STATUT DU PRODUCTEUR", "TYPE DE PRODUCTEUR"])
df_collection = df_collection.fillna("N/A")
df_collection["NATIF/ENRICHI"] = df_collection["NATIF/ENRICHI"].str.lower().replace("natf", "natif")
df_collection["Modèle économique"] = df_collection["Modèle économique"].str.lower().replace("gratut", "gratuit")

In [29]:
query = """
    INSERT INTO main_producteur(nom, status_producteur, type_producteur, model_eco, natif_enrichi) VALUES 
"""
for i, row in df_collection.iterrows(): 
    query += f"""('{str(row["NOM DU PRODUCTEUR"]).replace("'", "''")}', '{str(row["STATUT DU PRODUCTEUR"]).replace("'", "''")}', '{str(row["TYPE DE PRODUCTEUR"]).replace("'", "''")}', '{str(row["Modèle économique"]).replace("'", "''")}', '{str(row["NATIF/ENRICHI"]).replace("'", "''")}'),"""

query = query[:-1] + ";"

try : 
    execute_query(query, conn)
except : 
    conn = create_connection()
    execute_query(query, conn)


## podcast

In [30]:
def to_int(val) : 
    """
    This function take a number and try to convert it to int, if it"s work, function return int(val), else return -1
    """
    val = str(val)
    try : 
        val = int(val)
    except : 
        val = -1
    return val

def to_float(val) : 
    """
    This function take a number and try to convert it to float, if it"s work, function return float(val), else return -1
    """
    val = str(val)
    try : 
        val = float(val)
    except : 
        val = -1
    return val

def convert_date_for_pg(date_str: str) -> str:
    """
    Convertit une date JJ/MM/AAAA vers AAAA-MM-DD pour Postgres.
    Retourne '0001-01-01' si date invalide.
    """
    date_str = str(date_str).strip("/")
    
    try:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return dt.strftime("%Y-%m-%d")
    except Exception:
        return "0001-01-01"

def parse_two_values(s: str, function = int) -> tuple[int, float]:
    """
    Reçoit une chaîne 'int, float_avec_virgule'
    Exemple : "12, 4,4" -> (12, 4.4)
    Si format incorrect : retourne (0, 0)
    """
    s = str(s)

    try:
        # Split uniquement sur la première virgule
        parts = s.split(",", 1)
        if len(parts) != 2:
            return (0, 0)

        int_part = parts[0].strip()
        float_part = parts[1].strip().replace(",", ".")

        # Conversion
        i = int(int_part)
        f = func(float_part)

        return (i, f)

    except Exception:
        return (0, 0)

def parse_audience(value: str) -> int:
    """
    Parse les différentes formes d'audience et retourne un entier.
    Si la valeur est invalide → retourne 0.
    """

    value = str(value)

    s = value.strip().lower()

    # valeurs clairement invalides
    invalid = {"ad", "n", "non", "nsp", "", "na", "nan", "ne s'applique pas"}
    if s.lower() in invalid:
        return 0

    # Retirer texte entre parenthèses
    s = re.sub(r"\(.*?\)", "", s).strip()

    # Remplacer K / k par *1000
    s = s.replace("k", "000")

    # Remplacer espaces dans les nombres → "10 000" devient "10000"
    s = re.sub(r"(?<=\d)\s+(?=\d)", "", s)

    # Cas intervalle : "100-1500", "10 000 - 100 000", etc.
    if "-" in s or "à" in s or "a" in s:
        parts = re.split(r"-|à|a", s)
        nums = []
        for p in parts:
            p = p.strip()
            if p.isdigit():
                nums.append(int(p))
        if nums:
            return sum(nums) // len(nums)  # moyenne
        return 0

    # Cas simple : nombre direct
    s = s.replace(" ", "")
    if s.isdigit():
        return int(s)

    return 0


In [31]:
final_col = ["nom",
            "collection",
            "resume_officiel",
            "nom_host",
            "date_premier_episode",
            "date_derniere_episode",
            "datediff",
            "periodicite",
            "interrompu_moment_1_collection",
            "interrompu_moment_2_collection",
            "nb_episode_collecte_1",
            "nb_episode_collecte_2",
            "nb_episode_collecte_1_media",
            "nb_episode_collecte_2_media",
            "duree_moyenne",
            "duree_variable",
            "audience_site_nb_ecoute",
            "audience_apple_classement",
            "audience_apple_note",
            "audience_castbox_abonnement",
            "audience_castbox_nb_ecoute",
            "audience_podcastaddict_abonnement",
            "audience_youtube_nb_vue",
            "audience_soundcloud_nb_ecoute",
            "nb_telechargement_france",
            "nb_telechargement_monde",
            "date_collecte_1",
            "date_collecte_2",
            "producteur_id"
        ] 
cols = ["NOM", "COLLECTION", "Résumé officiel", "NOM DE L'HOST", "DATE DU PREMIER EPISODE",
        "DATE DU DERNIER EPISODE AU MOMENT DE LA COLLECTE ",
        "DATEDIF : nombre de mois d'écarts entre AC et AD", "PERIODICITE",
        "Interrompu au moment de la première collecte",
        "Interrompu lors de la deuxième collecte",
        "NOMBRE D'EPISODES (première collecte)",
        "Nombre d'épisodes (2e collecte)",
        "NOMBRE TOTAL D'ÉPISODES POUR LE MÉDIA première collecte",
        "nombre total d'épisode pour le média (2e collecte)",
        "DUREE MOYENNE ESTIMEES DES EPISODES (en minutes)", "Durée variable",
        
        "AUDIENCE - SITE (nombre d'écoutes)",
        "AUDIENCE - Apple Podcast nombre de classements", "AUDIENCE - Apple Podcast note", # add by me
        "AUDIENCE - Castbox nombre d'abonnements", "AUDIENCE - Castbox nombre d'écoutes",  # add by me
        "AUDIENCE PodcastAddict (nombre d'abonnements)",
        "AUDIENCE - Youtube (NOMBRE MOYEN DE vues)",
        "AUDIENCE - Soundcloud (nombre d'écoutes)",
        "ACPM - FÉVRIER 2022 -  TÈlÈchargements France sur 1 mois",
        "ACPM - FEVRIER 2022 - TÉLÉCHARGEMENTS MONDE SUR UN MOIS",
        
        "DATE DE LA Première COLLECTE", "Date de la deuxième collecte", 
        "id"  # added by me gor get id after join
       ]


In [32]:
df_podcast = df.fillna("N/A")

In [33]:
#les cols avant date sont ok 

#date premiere episode 
df_podcast["DATE DU PREMIER EPISODE"] = df_podcast["DATE DU PREMIER EPISODE"].apply(convert_date_for_pg)
df_podcast["DATE DU DERNIER EPISODE AU MOMENT DE LA COLLECTE "] = df_podcast["DATE DU DERNIER EPISODE AU MOMENT DE LA COLLECTE "].apply(convert_date_for_pg)
df_podcast["DATEDIF : nombre de mois d'écarts entre AC et AD"] = df_podcast["DATEDIF : nombre de mois d'écarts entre AC et AD"].apply(to_int)


df_podcast["NOMBRE D'EPISODES (première collecte)"] = df_podcast["NOMBRE D'EPISODES (première collecte)"].apply(to_int)
df_podcast["Nombre d'épisodes (2e collecte)"] = df_podcast["Nombre d'épisodes (2e collecte)"].apply(to_int)

df_podcast["NOMBRE TOTAL D'ÉPISODES POUR LE MÉDIA première collecte"] = df_podcast["NOMBRE TOTAL D'ÉPISODES POUR LE MÉDIA première collecte"].apply(to_int)
df_podcast["nombre total d'épisode pour le média (2e collecte)"] = df_podcast["nombre total d'épisode pour le média (2e collecte)"].apply(to_int)

df_podcast["DUREE MOYENNE ESTIMEES DES EPISODES (en minutes)"] = df_podcast["DUREE MOYENNE ESTIMEES DES EPISODES (en minutes)"].apply(to_float)

df_podcast["AUDIENCE - SITE (nombre d'écoutes)"] = df_podcast["AUDIENCE - SITE (nombre d'écoutes)"].apply(to_int)
df_podcast[["AUDIENCE - Apple Podcast nombre de classements", "AUDIENCE - Apple Podcast note"]] = df_podcast["AUDIENCE - Apple Podcast  (nombre de classements, note)"].apply(parse_two_values, function = float).apply(pd.Series)
df_podcast[["AUDIENCE - Castbox nombre d'abonnements", "AUDIENCE - Castbox nombre d'écoutes"]] = df_podcast["AUDIENCE - Castbox (nombre d'abonnements, nombre d'écoutes)"].apply(parse_two_values).apply(pd.Series)
df_podcast["AUDIENCE PodcastAddict (nombre d'abonnements)"] = df_podcast["AUDIENCE PodcastAddict (nombre d'abonnements)"].apply(to_int)

df_podcast["AUDIENCE - Youtube (NOMBRE MOYEN DE vues)"] = df_podcast["AUDIENCE - Youtube (NOMBRE MOYEN DE vues)"].apply(parse_audience)
df_podcast["AUDIENCE - Soundcloud (nombre d'écoutes)"] = df_podcast["AUDIENCE - Soundcloud (nombre d'écoutes)"].apply(to_int)

df_podcast["ACPM - FÉVRIER 2022 -  TÈlÈchargements France sur 1 mois"] = df_podcast["ACPM - FÉVRIER 2022 -  TÈlÈchargements France sur 1 mois"].apply(to_int)
df_podcast["ACPM - FEVRIER 2022 - TÉLÉCHARGEMENTS MONDE SUR UN MOIS"] = df_podcast["ACPM - FEVRIER 2022 - TÉLÉCHARGEMENTS MONDE SUR UN MOIS"].apply(to_int) 

df_podcast["DATE DE LA Première COLLECTE"] = df_podcast["DATE DE LA Première COLLECTE"].apply(convert_date_for_pg)
df_podcast["Date de la deuxième collecte"] = df_podcast["Date de la deuxième collecte"].apply(convert_date_for_pg)

In [34]:
# Lecture SQL
df_prod = pd.read_sql("SELECT * FROM main_producteur;", conn)

In [35]:
df_merged = df_podcast.merge(
    df_prod,
    left_on=["NOM DU PRODUCTEUR", "STATUT DU PRODUCTEUR", "TYPE DE PRODUCTEUR"],
    right_on=['nom', 'status_producteur', 'type_producteur'],
    how="left"
)

df_podcast = df_merged[cols]

In [36]:
df_podcast.shape

(2436, 29)

In [37]:
df_podcast.columns = final_col
engine = create_engine(f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}")

df_podcast.to_sql(
    "main_podcast",          # nom de la table dans PostgreSQL
    engine,
    if_exists="append", # "replace" / "append" / "fail"
    index=False          # ne pas écrire l'index comme colonne
)

print("DataFrame inséré dans PostgreSQL avec succès !")


DataFrame inséré dans PostgreSQL avec succès !


# peuplement table intermediaire

## podcast: mocle

In [38]:
def map_elem(val, liste) : 
    idx = []
    for i, j in liste: 
        if j in val : 
            idx.append(i)
    return idx


def peupler_join(source, column, new_name, destination, df= df, conn = conn, condition = ""): 
    df_left = pd.read_sql(f"SELECT * FROM {source} {condition};", conn)
    df_podcast = pd.read_sql("SELECT * FROM main_podcast;", conn)
    df_left_podcast = df.copy().fillna("na")
    
    
    liste = list(df_left[["id", "label"]].itertuples(index=False, name=None))
    
    
    df_left_podcast["idx"] = df_left_podcast[column].apply(map_elem, liste = liste)
    df_left_podcast = df_podcast.merge(df_left_podcast, left_on=["nom", "resume_officiel"], right_on=["NOM", "Résumé officiel"], how="left") 
    
    print("shape avant explode : ", df_left_podcast.shape)
    df_left_podcast = df_left_podcast[["id", "idx"]].explode("idx")
    
    df_left_podcast = df_left_podcast.drop_duplicates()
    print("shape apres explode drop duplicate : ", df_left_podcast.shape)
    
    df_left_podcast = df_left_podcast.dropna(axis=0)
    print("shape apres explode drop NA : ", df_left_podcast.shape)
    df_left_podcast.columns = ["podcast_id", new_name] 
    
    df_left_podcast.to_sql(destination, engine, if_exists="append", index=False)

In [39]:
peupler_join("main_motcle", "Mots-clefs", "motcle_id", "main_podcast_motcle")


shape avant explode :  (2438, 92)
shape apres explode drop duplicate :  (4037, 2)
shape apres explode drop NA :  (3389, 2)


## podcast : rubrique 

In [40]:
peupler_join("main_rubrique", "Rubrique", "rubrique_id", "main_podcast_rubrique")


shape avant explode :  (2438, 92)
shape apres explode drop duplicate :  (4018, 2)
shape apres explode drop NA :  (4011, 2)


## podcast recompense

In [41]:
peupler_join("main_recompense", "PRIX /RECOMPENSES", "recompense_id", "main_podcast_recompense")


shape avant explode :  (2438, 92)
shape apres explode drop duplicate :  (3025, 2)
shape apres explode drop NA :  (1687, 2)


## podcast : genre

In [42]:
peupler_join("main_genre", "Genre/format journalistique", "genre_id", "main_podcast_genre")

shape avant explode :  (2438, 92)
shape apres explode drop duplicate :  (2438, 2)
shape apres explode drop NA :  (1196, 2)


## podcast : etiquette

In [43]:
peupler_join("main_etiquette", "SPOTIFY ETIQUETTES THEMATIQUES", "etiquette_id", "main_podcast_etiquette", condition="WHERE type_etiquette='SPOTIFY'")
peupler_join("main_etiquette", "APPLE PODCAST ETIQUETTES THEMATIQUES", "etiquette_id", "main_podcast_etiquette", condition="WHERE type_etiquette='APPLE'")

shape avant explode :  (2438, 92)
shape apres explode drop duplicate :  (3309, 2)
shape apres explode drop NA :  (2968, 2)
shape avant explode :  (2438, 92)
shape apres explode drop duplicate :  (2722, 2)
shape apres explode drop NA :  (1969, 2)


## podcast : createur

In [44]:
df_createur_sql = pd.read_sql("SELECT * FROM main_createur;", conn)
df_createur_sql = df_createur_sql.merge(df_list_createur_unique, on=["nom", "genre"], how="left")
df_createur_sql = df_createur_sql.dropna()
df_createur_sql["index"] = (df_createur_sql["index"] + 1).apply(lambda x: int(x))
df_createur_sql = df_createur_sql[["index", "id"]].drop_duplicates()
df_createur_sql.columns = ["podcast_id", "createur_id"]
df_createur_sql.to_sql("main_podcast_createur", engine, if_exists="append", index=False)

724

## podcast : plateforme

In [45]:
plateforme =  ["id"] + [x for x in df.columns if "disponible sur" in x] 
df_podcast_with_id = pd.read_sql("SELECT * FROM main_podcast;", conn)
df_podcast_with_id = df_podcast_with_id.merge(df, left_on=["nom", "resume_officiel"], right_on=["NOM", "Résumé officiel"], how="left")[plateforme]

In [46]:
df_podcast_with_id.head()

,id,SITE DU PRODUCTEUR disponible sur,APPLE PODCAST disponible sur,SPOTIFY disponible sur,DEEZER disponible sur,GOOGLE PODCAST disponible sur,CASTBOX disponible sur,PODCAST ADDICT disponible sur,YOUTUBE disponible sur,SOUNDCLOUD disponible sur,SITE PARTENAIRE disponible sur,STITCHER disponible sur,AMAZON MUSIC disponible sur
0,1,OUI,NON,NON,NON,NON,NON,NON,NON,NON,NON,NON,non
1,2,oui,oui,non,AD,AD,AD,AD,AD,AD,AD,non,AD
2,3,oui,non,non,AD,AD,AD,AD,AD,AD,AD,non,AD
3,4,oui,oui,oui,AD,AD,AD,AD,AD,AD,AD,non,AD
4,5,oui,non,non,AD,AD,AD,AD,AD,AD,AD,non,AD


In [47]:
df_plateforme__podcast = {"podcast_id" : [], "col" : [], "value" : []}
for _, row in df_podcast_with_id.iterrows(): 
    cols = list(row.index)
    cols.remove("id")

    for col in cols : 
        if isinstance(row[col], str): 
            x = col.split("disponible sur")[0].strip()
            df_plateforme__podcast["podcast_id"].append(row["id"])
            df_plateforme__podcast["col"].append(x)
            df_plateforme__podcast["value"].append(row[col].strip().lower())


df_plateforme__podcast = pd.DataFrame(df_plateforme__podcast)
df_plateforme = pd.read_sql("SELECT * FROM main_plateforme;", conn )
df_plateforme__podcast = df_plateforme__podcast.merge(df_plateforme, left_on="col", right_on="label", how="left")[["podcast_id", "id", "value"]]
df_plateforme__podcast.columns = ["podcast_id", "plateforme_id", "etat"]
df_plateforme__podcast = df_plateforme__podcast.drop_duplicates(subset=["podcast_id", "plateforme_id"])
df_plateforme__podcast.value_counts()

podcast_id  plateforme_id  etat
1           1              oui     1
1625        8              non     1
            6              non     1
            5              oui     1
            4              oui     1
                                  ..
812         9              oui     1
            8              non     1
            7              oui     1
            6              oui     1
2436        10             ad      1
Name: count, Length: 29230, dtype: int64

In [48]:
df_plateforme__podcast.to_sql("main_podcast_plateforme", engine, if_exists="append", index=False)

230

# conserve